In [ ]:
!git clone https://github.com/MR-just01/Llama3.2-Reasoning

%cd Llama3.2-Reasoning

!find . -maxdepth 3 -type f | head -50

In [ ]:
import pandas as pd

validation_df = pd.read_csv(
    "./data/processed/validation_results.csv"
)

print("Rows:", len(validation_df))
print("Columns:")
print(validation_df.columns.tolist())

display(validation_df.head())

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("No GPU detected.")

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
!pip install -q groq

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

GROQ_API_KEY = user_secrets.get_secret(
    "GROQ_API_KEY"
)

print("API key loaded:", bool(GROQ_API_KEY))

In [ ]:
from groq import Groq

client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq client initialized.")

In [ ]:
import transformers
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

In [ ]:
from huggingface_hub import login

login()

In [ ]:
print(validation_df.info())

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

print("✅ Groq API key loaded successfully")
print("Key length:", len(GROQ_API_KEY))

In [ ]:
import os

print("GROQ_API_KEY exists:",
      "GROQ_API_KEY" in os.environ)

print("Possible Groq variables:")

for key in os.environ:
    if "GROQ" in key.upper():
        print(key)

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "What is 15 × 3? Return only the number."
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

In [ ]:
# ============================================================
# FINAL LLAMA EVALUATION PIPELINE
# CELL 1 — SETUP + FIXED STRATIFIED 300-ROW SAMPLE
# ============================================================

import os
import json
import time
import math
import random
import numpy as np
import pandas as pd
from groq import Groq


# ============================================================
# CONFIG
# ============================================================

GROQ_MODEL = "openai/gpt-oss-120b"

SAMPLE_SIZE = 300
CALIBRATION_SIZE = 10

RANDOM_STATE = 42

CHECKPOINT_PATH = (
    "/kaggle/working/llama_groq_final_300_checkpoint.csv"
)

SAMPLE_PATH = (
    "/kaggle/working/llama_groq_final_300_sample.csv"
)

CALIBRATION_PATH = (
    "/kaggle/working/llama_groq_calibration_10.csv"
)

# High reasoning effort for calibration.
CALIBRATION_REASONING_EFFORT = "high"

# We will decide the full-evaluation effort after
# manually auditing the calibration results.
FULL_REASONING_EFFORT = "medium"

MAX_RETRIES = 3
INITIAL_RETRY_WAIT = 5

MAX_COMPLETION_TOKENS = 1200

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# ============================================================
# GROQ CLIENT
# ============================================================

# GROQ_API_KEY was loaded previously from Kaggle Secrets.

client = Groq(
    api_key=GROQ_API_KEY
)

print("✅ Groq client initialized")
print("Judge model:", GROQ_MODEL)


# ============================================================
# PREPARE EVALUATION DATA
# ============================================================

evaluation_df = validation_df[
    validation_df["expected_answer"].notna()
    &
    validation_df["input"].notna()
    &
    validation_df["model_response"].notna()
].copy()

evaluation_df["index"] = (
    evaluation_df["index"].astype(str)
)

print()
print("=" * 80)
print("ELIGIBLE EVALUATION ROWS:", len(evaluation_df))
print("=" * 80)


# ============================================================
# BASIC SAFETY CHECK
# ============================================================

if len(evaluation_df) < SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(evaluation_df)} eligible rows are available, "
        f"but SAMPLE_SIZE={SAMPLE_SIZE}."
    )


# ============================================================
# CREATE / LOAD FIXED STRATIFIED SAMPLE
# ============================================================

if os.path.exists(SAMPLE_PATH):

    # --------------------------------------------------------
    # Load previously created fixed sample
    # --------------------------------------------------------

    sample_df = pd.read_csv(SAMPLE_PATH)

    sample_df["index"] = (
        sample_df["index"].astype(str)
    )

    print("\nExisting fixed sample loaded.")
    print("Sample rows:", len(sample_df))


else:

    # --------------------------------------------------------
    # Determine stratification variable
    # --------------------------------------------------------

    if "task_type" in evaluation_df.columns:

        evaluation_df["_stratum"] = (
            evaluation_df["task_type"]
            .fillna("UNKNOWN")
            .astype(str)
        )

        stratification_used = "task_type"

    elif "dataset" in evaluation_df.columns:

        evaluation_df["_stratum"] = (
            evaluation_df["dataset"]
            .fillna("UNKNOWN")
            .astype(str)
        )

        stratification_used = "dataset"

    else:

        evaluation_df["_stratum"] = "ALL"

        stratification_used = "none"


    print(
        "Stratification:",
        stratification_used
    )


    # --------------------------------------------------------
    # Calculate group sizes
    # --------------------------------------------------------

    group_sizes = (
        evaluation_df["_stratum"]
        .value_counts()
        .sort_index()
    )

    # Proportional allocation before rounding
    raw_allocations = (
        group_sizes
        / len(evaluation_df)
        * SAMPLE_SIZE
    )


    # --------------------------------------------------------
    # Floor allocations
    #
    # IMPORTANT:
    # np.floor() is used instead of
    # raw_allocations.floor()
    # because raw_allocations is a Pandas Series.
    # --------------------------------------------------------

    allocations = (
        np.floor(raw_allocations)
        .astype(int)
    )


    # --------------------------------------------------------
    # Largest-remainder method
    #
    # Distribute rows lost during flooring.
    # --------------------------------------------------------

    remaining_slots = (
        SAMPLE_SIZE
        - allocations.sum()
    )

    remainders = (
        raw_allocations - allocations
    ).sort_values(
        ascending=False
    )


    for group in remainders.index:

        if remaining_slots <= 0:
            break

        # Do not allocate more rows than the group contains.
        if allocations.loc[group] < group_sizes.loc[group]:

            allocations.loc[group] += 1
            remaining_slots -= 1


    print("\nStratified allocations:")
    print(allocations)

    print(
        "\nTotal allocated after rounding:",
        allocations.sum()
    )


    # --------------------------------------------------------
    # Sample from each stratum
    # --------------------------------------------------------

    sampled_parts = []

    for group, n in allocations.items():

        if n <= 0:
            continue

        group_df = evaluation_df[
            evaluation_df["_stratum"] == group
        ]

        # Safety: never sample more rows than available.
        n = min(
            int(n),
            len(group_df)
        )

        if n == 0:
            continue

        sampled_group = group_df.sample(
            n=n,
            random_state=RANDOM_STATE
        )

        sampled_parts.append(
            sampled_group
        )


    # --------------------------------------------------------
    # Combine sampled groups
    # --------------------------------------------------------

    if sampled_parts:

        sample_df = pd.concat(
            sampled_parts,
            ignore_index=True
        )

    else:

        raise RuntimeError(
            "No rows were sampled. "
            "Check the stratification column."
        )


    # --------------------------------------------------------
    # Safety fill
    #
    # If group capacity or rounding caused fewer than
    # SAMPLE_SIZE rows, fill from unused eligible rows.
    # --------------------------------------------------------

    if len(sample_df) < SAMPLE_SIZE:

        selected_indices = set(
            sample_df["index"].astype(str)
        )

        remaining_candidates = evaluation_df[
            ~evaluation_df["index"].isin(
                selected_indices
            )
        ]

        extra_needed = (
            SAMPLE_SIZE - len(sample_df)
        )

        if len(remaining_candidates) < extra_needed:

            raise RuntimeError(
                "Not enough unused eligible rows to "
                "complete the requested sample."
            )

        extra = remaining_candidates.sample(
            n=extra_needed,
            random_state=RANDOM_STATE
        )

        sample_df = pd.concat(
            [
                sample_df,
                extra
            ],
            ignore_index=True
        )


    # --------------------------------------------------------
    # Safety trim
    # --------------------------------------------------------

    elif len(sample_df) > SAMPLE_SIZE:

        sample_df = sample_df.sample(
            n=SAMPLE_SIZE,
            random_state=RANDOM_STATE
        ).reset_index(drop=True)


    # --------------------------------------------------------
    # Remove helper column
    # --------------------------------------------------------

    sample_df = sample_df.drop(
        columns=["_stratum"],
        errors="ignore"
    )


    # --------------------------------------------------------
    # Shuffle final sample
    # --------------------------------------------------------

    sample_df = (
        sample_df
        .sample(
            frac=1,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Save fixed sample
    # --------------------------------------------------------

    sample_df.to_csv(
        SAMPLE_PATH,
        index=False
    )

    print(
        "\nNew fixed 300-row sample created."
    )


# ============================================================
# VERIFY SAMPLE
# ============================================================

sample_df["index"] = (
    sample_df["index"].astype(str)
)


# Exact sample-size check
assert len(sample_df) == SAMPLE_SIZE, (
    f"Expected {SAMPLE_SIZE} rows, "
    f"got {len(sample_df)}"
)


# No duplicate evaluation indices
assert (
    sample_df["index"].nunique()
    == SAMPLE_SIZE
), "Duplicate indices detected in sample."


# Required columns
required_columns = [
    "index",
    "input",
    "expected_answer",
    "model_response"
]

missing_columns = [
    col
    for col in required_columns
    if col not in sample_df.columns
]

if missing_columns:

    raise ValueError(
        f"Sample is missing required columns: "
        f"{missing_columns}"
    )


# ============================================================
# FINAL SAMPLE INFORMATION
# ============================================================

print()
print("=" * 80)
print("FIXED STRATIFIED SAMPLE")
print("=" * 80)

print("Rows:", len(sample_df))

print(
    "Unique indices:",
    sample_df["index"].nunique()
)

print(
    "Saved to:",
    SAMPLE_PATH
)

print("\nFirst 10 indices:")

print(
    sample_df["index"]
    .head(10)
    .tolist()
)

print("\nSample ready for calibration.")

In [ ]:
# ============================================================
# CELL 2 — GROQ JUDGE FUNCTION
# GPT-OSS-120B
# ============================================================

import json
import re
import time


# ============================================================
# JUDGE PROMPT
# ============================================================

def build_judge_prompt(row):

    question = str(row["input"])
    reference_answer = str(row["expected_answer"])
    llama_response = str(row["model_response"])

    prompt = f"""
You are an expert evaluator judging the response of a fine-tuned Llama model.

Your task is to independently evaluate:

1. Whether the QUESTION is valid and solvable.
2. Whether the REFERENCE ANSWER is correct for that question.
3. Whether the Llama's FINAL ANSWER is correct.
4. Whether the Llama's REASONING is logically and mathematically valid.

IMPORTANT:

Do NOT judge the Llama answer merely by comparing text strings.

You must understand the question and independently determine the correct result.

Also do NOT assume that the Llama reasoning is correct just because its final answer is correct.

A correct final answer with invalid reasoning must receive:

"answer_correct": true
"reasoning_correct": false

Likewise, an incorrect final answer with otherwise meaningful reasoning must receive:

"answer_correct": false

You must evaluate the reasoning itself.

IMPORTANT ORDER:

First analyze the problem and the Llama response carefully.

Then produce your final structured judgment.

Do not guess the verdict before analyzing the response.

------------------------------------------------------------
QUESTION
------------------------------------------------------------

{question}

------------------------------------------------------------
REFERENCE ANSWER
------------------------------------------------------------

{reference_answer}

------------------------------------------------------------
LLAMA RESPONSE
------------------------------------------------------------

{llama_response}

------------------------------------------------------------
EVALUATION
------------------------------------------------------------

Determine:

- Is the question valid?
- Is the reference answer correct?
- Is the Llama final answer correct?
- Is the Llama reasoning correct?

If the question is invalid or the reference answer is wrong, explain that clearly.

For the reasoning judgment, check whether the actual steps used by Llama are logically valid.
Do not punish Llama merely for being brief.
Do not require the exact wording of the reference solution.
Equivalent mathematical reasoning is acceptable.

Return ONLY one JSON object after completing your analysis.

The JSON must contain exactly these fields:

{{
  "question_valid": true,
  "reference_answer_correct": true,
  "answer_correct": true,
  "reasoning_correct": true,
  "judge_evidence": "brief explanation supporting the verdicts"
}}

The values for the first four fields must be true or false.

"judge_evidence" must be a concise explanation of the actual evidence used to reach the verdicts.

Do not put markdown around the JSON.
"""


    return prompt


# ============================================================
# JSON EXTRACTION
# ============================================================

def extract_json_object(text):

    if text is None:
        return None

    text = str(text).strip()

    # --------------------------------------------------------
    # First try the entire response
    # --------------------------------------------------------

    try:
        parsed = json.loads(text)

        if isinstance(parsed, dict):
            return parsed

    except Exception:
        pass


    # --------------------------------------------------------
    # Try extracting the last JSON object
    # --------------------------------------------------------

    matches = re.findall(
        r'\{(?:[^{}]|(?:\{[^{}]*\}))*\}',
        text,
        flags=re.DOTALL
    )

    for candidate in reversed(matches):

        try:

            parsed = json.loads(candidate)

            if isinstance(parsed, dict):
                return parsed

        except Exception:
            continue


    return None


# ============================================================
# SINGLE-ROW JUDGE
# ============================================================

def judge_one_row(
    row,
    reasoning_effort="high",
    max_retries=3,
    initial_retry_wait=5
):

    prompt = build_judge_prompt(row)

    last_error = None

    for attempt in range(max_retries):

        start_time = time.time()

        try:

            response = client.chat.completions.create(

                model=GROQ_MODEL,

                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a rigorous LLM evaluator. "
                            "Analyze the response before producing "
                            "your final JSON judgment."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],

                # IMPORTANT:
                # We intentionally use HIGH for calibration.
                reasoning_effort=reasoning_effort,

                # Do NOT use:
                # reasoning_format="hidden"

                max_completion_tokens=MAX_COMPLETION_TOKENS,

                temperature=0
            )


            # ------------------------------------------------
            # Extract response
            # ------------------------------------------------

            raw_content = (
                response
                .choices[0]
                .message
                .content
            )

            elapsed = time.time() - start_time


            # ------------------------------------------------
            # Parse JSON
            # ------------------------------------------------

            parsed = extract_json_object(
                raw_content
            )


            if parsed is None:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "FAILED",
                    "raw_output": raw_content,
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Validate required fields
            # ------------------------------------------------

            required_fields = [
                "question_valid",
                "reference_answer_correct",
                "answer_correct",
                "reasoning_correct",
                "judge_evidence"
            ]

            missing_fields = [
                field
                for field in required_fields
                if field not in parsed
            ]


            if missing_fields:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": (
                        f"Missing fields: {missing_fields}"
                    ),
                    "parse_status": "INVALID_SCHEMA",
                    "raw_output": raw_content,
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Normalize boolean fields
            # ------------------------------------------------

            boolean_fields = [
                "question_valid",
                "reference_answer_correct",
                "answer_correct",
                "reasoning_correct"
            ]

            for field in boolean_fields:

                value = parsed[field]

                if isinstance(value, str):

                    value_lower = value.lower().strip()

                    if value_lower == "true":
                        parsed[field] = True

                    elif value_lower == "false":
                        parsed[field] = False

                    else:
                        parsed[field] = None


            # ------------------------------------------------
            # Return successful result
            # ------------------------------------------------

            return {
                "question_valid": parsed[
                    "question_valid"
                ],

                "reference_answer_correct": parsed[
                    "reference_answer_correct"
                ],

                "answer_correct": parsed[
                    "answer_correct"
                ],

                "reasoning_correct": parsed[
                    "reasoning_correct"
                ],

                "judge_evidence": str(
                    parsed["judge_evidence"]
                ),

                "parse_status": "OK",

                "raw_output": raw_content,

                "time_seconds": round(
                    elapsed,
                    2
                )
            }


        except Exception as e:

            last_error = str(e)

            # Retry with exponential backoff
            if attempt < max_retries - 1:

                wait_time = (
                    initial_retry_wait
                    * (2 ** attempt)
                )

                print(
                    f"Retry {attempt + 1}/{max_retries - 1} "
                    f"after error: {last_error}"
                )

                time.sleep(wait_time)


    # ========================================================
    # ALL RETRIES FAILED
    # ========================================================

    return {
        "question_valid": None,
        "reference_answer_correct": None,
        "answer_correct": None,
        "reasoning_correct": None,
        "judge_evidence": None,
        "parse_status": "ERROR",
        "raw_output": "",
        "error": last_error,
        "time_seconds": None
    }


# ============================================================
# VERIFY FUNCTION EXISTS
# ============================================================

print("=" * 80)
print("JUDGE FUNCTION READY")
print("=" * 80)

print("Model:", GROQ_MODEL)
print("Function:", judge_one_row.__name__)
print("Calibration reasoning:", CALIBRATION_REASONING_EFFORT)
print("Full evaluation reasoning:", FULL_REASONING_EFFORT)
print("Max completion tokens:", MAX_COMPLETION_TOKENS)

print("\n✅ Cell 2 loaded successfully.")

In [ ]:
# ============================================================
# GROQ RATE-LIMIT TEST
# ============================================================

import time

print("=" * 80)
print("TESTING GROQ GPT-OSS-120B AVAILABILITY")
print("=" * 80)

try:
    start = time.time()

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": "Reply with exactly: OK"
            }
        ],
        reasoning_effort="low",
        max_completion_tokens=10
    )

    elapsed = time.time() - start

    content = response.choices[0].message.content

    print("✅ REQUEST SUCCESSFUL")
    print("Response:", repr(content))
    print("Time:", round(elapsed, 2), "seconds")

    if hasattr(response, "usage"):
        print("\nUsage:")
        print(response.usage)

except Exception as e:

    print("❌ REQUEST FAILED")
    print("Error type:", type(e).__name__)
    print("Error:")
    print(str(e))

In [ ]:
# ============================================================
# DIRECT GPT-OSS-120B DIAGNOSTIC
# ============================================================

row = sample_df[
    sample_df["index"].astype(str) == "11204"
].iloc[0]

prompt = build_judge_prompt(row)

print("=" * 80)
print("DIRECT GPT-OSS-120B DIAGNOSTIC")
print("=" * 80)

try:

    start = time.time()

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",

        messages=[
            {
                "role": "system",
                "content": (
                    "You are a rigorous evaluator. "
                    "Analyze the problem before giving "
                    "your final JSON judgment."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        reasoning_effort="high",

        max_completion_tokens=2000,

        temperature=0
    )

    elapsed = time.time() - start

    print("\nREQUEST SUCCESSFUL")
    print("Time:", round(elapsed, 2), "seconds")

    print("\nRAW RESPONSE OBJECT:")
    print(response)

    print("\nMESSAGE CONTENT:")
    print(
        repr(
            response.choices[0].message.content
        )
    )

    print("\nFINISH REASON:")
    print(
        response.choices[0].finish_reason
    )

    print("\nUSAGE:")
    print(response.usage)

except Exception as e:

    print("\nREQUEST FAILED")
    print("Exception:", type(e).__name__)
    print(str(e))

In [ ]:
# ============================================================
# GPT-OSS-120B JUDGE V3
# HIGH REASONING + SIMPLE JSON OUTPUT
# ============================================================

def judge_one_row(
    row,
    reasoning_effort="high",
    max_retries=2,
    initial_retry_wait=15
):

    question = str(row["input"])
    reference_answer = str(row["expected_answer"])
    llama_response = str(row["model_response"])

    prompt = f"""
You are evaluating a fine-tuned Llama language model.

Carefully solve the question yourself first. Then evaluate the
Llama response.

QUESTION:
{question}

REFERENCE ANSWER:
{reference_answer}

LLAMA RESPONSE:
{llama_response}

Evaluate these four things:

1. question_valid
   Is the question valid and sufficiently specified?

2. reference_answer_correct
   Is the reference answer actually correct?

3. answer_correct
   Is Llama's final answer correct?

4. reasoning_correct
   Is Llama's reasoning logically or mathematically valid?

Rules:

- Do NOT use string matching alone.
- Independently solve the question.
- A correct final answer does NOT imply correct reasoning.
- If the final answer is correct but the reasoning is invalid:
  answer_correct = true
  reasoning_correct = false
- If Llama provides no reasoning where reasoning is required:
  reasoning_correct = false.
- Equivalent valid reasoning is acceptable.
- If the question itself is invalid, reflect that in question_valid.
- If the reference answer is wrong, reflect that in
  reference_answer_correct.

Think through the problem carefully before producing the verdict.

At the very end, output ONLY this JSON object:

{{
  "question_valid": true,
  "reference_answer_correct": true,
  "answer_correct": true,
  "reasoning_correct": true
}}

Do not output anything before or after the JSON.
"""


    for attempt in range(max_retries):

        start_time = time.time()

        try:

            response = client.chat.completions.create(

                model=GROQ_MODEL,

                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],

                reasoning_effort=reasoning_effort,

                # Give the reasoning model enough room.
                max_completion_tokens=5000,

                temperature=0
            )


            elapsed = time.time() - start_time

            message = response.choices[0].message

            raw_content = (
                message.content or ""
            ).strip()

            finish_reason = (
                response.choices[0].finish_reason
            )


            # ------------------------------------------------
            # Empty output
            # ------------------------------------------------

            if not raw_content:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "EMPTY_RESPONSE",
                    "raw_output": raw_content,
                    "finish_reason": finish_reason,
                    "error": (
                        "Model finished without visible JSON."
                    ),
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Extract JSON
            # ------------------------------------------------

            parsed = extract_json_object(
                raw_content
            )


            if parsed is None:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "PARSE_FAILED",
                    "raw_output": raw_content,
                    "finish_reason": finish_reason,
                    "error": (
                        "Could not extract JSON "
                        "from model output."
                    ),
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Validate fields
            # ------------------------------------------------

            required_fields = [
                "question_valid",
                "reference_answer_correct",
                "answer_correct",
                "reasoning_correct"
            ]

            missing = [
                field
                for field in required_fields
                if field not in parsed
            ]

            if missing:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "INVALID_SCHEMA",
                    "raw_output": raw_content,
                    "finish_reason": finish_reason,
                    "error": (
                        f"Missing fields: {missing}"
                    ),
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Successful judgment
            # ------------------------------------------------

            return {
                "question_valid": bool(
                    parsed["question_valid"]
                ),

                "reference_answer_correct": bool(
                    parsed[
                        "reference_answer_correct"
                    ]
                ),

                "answer_correct": bool(
                    parsed["answer_correct"]
                ),

                "reasoning_correct": bool(
                    parsed["reasoning_correct"]
                ),

                # Evidence is intentionally empty here.
                # We can audit the raw reasoning separately.
                "judge_evidence": None,

                "parse_status": "OK",

                "raw_output": raw_content,

                "finish_reason": finish_reason,

                "time_seconds": round(
                    elapsed,
                    2
                )
            }


        except Exception as e:

            error_text = str(e)

            print(
                f"Attempt {attempt + 1}/{max_retries} failed:"
            )
            print(error_text)


            if attempt < max_retries - 1:

                wait_time = (
                    initial_retry_wait
                    * (2 ** attempt)
                )

                print(
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "API_ERROR",
                    "raw_output": "",
                    "finish_reason": None,
                    "error": error_text,
                    "time_seconds": None
                }

In [ ]:
test_index = "11204"

test_row = sample_df[
    sample_df["index"].astype(str) == test_index
].iloc[0]

result_11204 = judge_one_row(
    test_row,
    reasoning_effort="high",
    max_retries=1
)

print("=" * 80)
print("11204 TEST")
print("=" * 80)

print("Question valid:", result_11204["question_valid"])
print(
    "Reference correct:",
    result_11204["reference_answer_correct"]
)
print(
    "Llama answer correct:",
    result_11204["answer_correct"]
)
print(
    "Llama reasoning correct:",
    result_11204["reasoning_correct"]
)

print(
    "Parse:",
    result_11204["parse_status"]
)

print(
    "Finish reason:",
    result_11204.get("finish_reason")
)

print(
    "Time:",
    result_11204.get("time_seconds")
)

print(
    "\nRaw JSON:"
)

print(
    result_11204["raw_output"]
)

if result_11204.get("error"):
    print("\nERROR:")
    print(result_11204["error"])

In [ ]:
test_index = "27275"

test_row = sample_df[
    sample_df["index"].astype(str) == test_index
].iloc[0]

result_27275 = judge_one_row(
    test_row,
    reasoning_effort="high",
    max_retries=1
)

print("=" * 80)
print("27275 TEST")
print("=" * 80)

print("Question valid:", result_27275["question_valid"])
print(
    "Reference correct:",
    result_27275["reference_answer_correct"]
)
print(
    "Llama answer correct:",
    result_27275["answer_correct"]
)
print(
    "Llama reasoning correct:",
    result_27275["reasoning_correct"]
)

print(
    "Parse:",
    result_27275["parse_status"]
)

print(
    "Finish reason:",
    result_27275.get("finish_reason")
)

print(
    "Time:",
    result_27275.get("time_seconds")
)

print("\nRaw JSON:")
print(result_27275["raw_output"])

if result_27275.get("error"):
    print("\nERROR:")
    print(result_27275["error"])

In [ ]:
# ============================================================
# SHOW THE ACTUAL API ERROR FROM 11204
# ============================================================

print("=" * 80)
print("11204 API ERROR")
print("=" * 80)

print("Error:")
print(result_11204.get("error"))

print("\nParse status:")
print(result_11204.get("parse_status"))

print("\nFinish reason:")
print(result_11204.get("finish_reason"))

In [ ]:
# ============================================================
# BUILD FINAL 10-ROW CALIBRATION
# USING THE TWO SUCCESSFUL RETRIES ALREADY IN MEMORY
# ============================================================

# Create retry_success directly from the two results
retry_success = pd.DataFrame([
    {
        "index": "11204",
        **result_11204
    },
    {
        "index": "27275",
        **result_27275
    }
])

# Keep only successful judgments
retry_success = retry_success[
    retry_success["parse_status"] == "OK"
].copy()

# Make index type consistent
retry_success["index"] = (
    retry_success["index"].astype(str)
)

print("=" * 80)
print("SUCCESSFUL RETRIES")
print("=" * 80)

print("Rows:", len(retry_success))
print("Indices:", retry_success["index"].tolist())


# ============================================================
# ORIGINAL 8 SUCCESSFUL CALIBRATION ROWS
# ============================================================

original_success = calibration_results_df[
    calibration_results_df["parse_status"] == "OK"
].copy()

original_success["index"] = (
    original_success["index"].astype(str)
)


# ============================================================
# COMBINE
# ============================================================

final_calibration_df = pd.concat(
    [
        original_success,
        retry_success
    ],
    ignore_index=True
)


# Remove duplicate indices if any
final_calibration_df = (
    final_calibration_df
    .drop_duplicates(
        subset=["index"],
        keep="last"
    )
    .reset_index(drop=True)
)


# ============================================================
# VERIFY
# ============================================================

print()
print("=" * 80)
print("FINAL CALIBRATION DATASET")
print("=" * 80)

print("Rows:", len(final_calibration_df))

print(
    "Unique indices:",
    final_calibration_df["index"].nunique()
)

print("\nIndices:")
print(
    final_calibration_df["index"].tolist()
)


print("\nQuestion validity:")
print(
    final_calibration_df[
        "question_valid"
    ].value_counts(dropna=False)
)


print("\nReference answer validity:")
print(
    final_calibration_df[
        "reference_answer_correct"
    ].value_counts(dropna=False)
)


print("\nLlama answer correctness:")
print(
    final_calibration_df[
        "answer_correct"
    ].value_counts(dropna=False)
)


print("\nLlama reasoning correctness:")
print(
    final_calibration_df[
        "reasoning_correct"
    ].value_counts(dropna=False)
)


print("\nParse status:")
print(
    final_calibration_df[
        "parse_status"
    ].value_counts(dropna=False)
)


# ============================================================
# SAVE FINAL CALIBRATION
# ============================================================

CALIBRATION_FINAL_PATH = (
    "/kaggle/working/"
    "llama_groq_final_calibration_10.csv"
)

final_calibration_df.to_csv(
    CALIBRATION_FINAL_PATH,
    index=False
)

print()
print("=" * 80)
print("CALIBRATION SAVED")
print("=" * 80)

print(CALIBRATION_FINAL_PATH)

In [ ]:
import os
import glob

print("=" * 70)
print("SEARCHING FOR EXISTING EVALUATION FILES")
print("=" * 70)

patterns = [
    "*checkpoint*",
    "*groq*",
    "*300*",
    "*evaluation*"
]

found = set()

for pattern in patterns:
    for path in glob.glob(
        f"/kaggle/working/{pattern}",
        recursive=False
    ):
        found.add(path)

for path in sorted(found):
    print(path)

print("\nTotal files found:", len(found))

In [ ]:
# ============================================================
# FINAL RESUMABLE 300-ROW EVALUATION
#
# RULES:
# 1. NEVER create a new sample
# 2. Use the existing fixed 300-row sample
# 3. ONLY parse_status == "OK" counts as completed
# 4. Save after EVERY successful row
# 5. Failed rows are NOT marked completed
# 6. Stop immediately on rate/quota limit
# 7. Resume from checkpoint on next run
# 8. Use atomic checkpoint writes
# ============================================================

import os
import time
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

EVAL_SIZE = 300

REASONING_EFFORT = "high"

MAX_RETRIES = 2
RETRY_WAIT = 20

# ------------------------------------------------------------
# IMPORTANT: THESE FILENAMES MUST STAY CONSISTENT
# ------------------------------------------------------------

RANDOM_SAMPLE_PATH = (
    "/kaggle/working/"
    "llama_groq_final_300_sample.csv"
)

CHECKPOINT_PATH = (
    "/kaggle/working/"
    "llama_groq_final_300_checkpoint.csv"
)

FAILURE_LOG_PATH = (
    "/kaggle/working/"
    "llama_groq_final_300_failures.csv"
)

FINAL_RESULTS_PATH = (
    "/kaggle/working/"
    "llama_groq_final_300_results.csv"
)

CALIBRATION_PATH = (
    "/kaggle/working/"
    "llama_groq_final_calibration_10.csv"
)


# ============================================================
# HELPER: ATOMIC CSV SAVE
# ============================================================

def atomic_save_csv(df, path):

    temp_path = path + ".tmp"

    df.to_csv(
        temp_path,
        index=False
    )

    os.replace(
        temp_path,
        path
    )


# ============================================================
# HELPER: RATE LIMIT DETECTION
# ============================================================

def is_rate_limit_error(error_text):

    text = str(error_text).lower()

    indicators = [
        "429",
        "rate limit",
        "rate_limit",
        "rate-limit",
        "quota",
        "too many requests",
        "tokens per day",
        "tpd",
        "limit reached",
        "limit exceeded"
    ]

    return any(
        indicator in text
        for indicator in indicators
    )


# ============================================================
# 1. LOAD FIXED SAMPLE
# ============================================================

if not os.path.exists(RANDOM_SAMPLE_PATH):

    raise FileNotFoundError(
        "\nFixed 300-row sample not found:\n"
        f"{RANDOM_SAMPLE_PATH}\n\n"
        "DO NOT create a new sample.\n"
        "The original fixed sample must be restored first."
    )


sample_df = pd.read_csv(
    RANDOM_SAMPLE_PATH
)

sample_df["index"] = (
    sample_df["index"]
    .astype(str)
)


# ============================================================
# 2. VERIFY SAMPLE
# ============================================================

if len(sample_df) != EVAL_SIZE:

    raise ValueError(
        f"Expected exactly {EVAL_SIZE} rows, "
        f"but found {len(sample_df)}."
    )


if sample_df["index"].nunique() != EVAL_SIZE:

    raise ValueError(
        "Duplicate indices detected in fixed sample."
    )


sample_indices = set(
    sample_df["index"]
)


print("=" * 80)
print("FIXED 300-ROW SAMPLE")
print("=" * 80)

print(
    "Rows:",
    len(sample_df)
)

print(
    "Unique indices:",
    sample_df["index"].nunique()
)

print(
    "Sample:",
    RANDOM_SAMPLE_PATH
)


# ============================================================
# 3. OPTIONAL: VERIFY CALIBRATION ROWS ARE NOT IN SAMPLE
# ============================================================

if os.path.exists(CALIBRATION_PATH):

    calibration_df = pd.read_csv(
        CALIBRATION_PATH
    )

    if "index" in calibration_df.columns:

        calibration_indices = set(
            calibration_df["index"]
            .astype(str)
        )

        overlap = (
            sample_indices
            .intersection(
                calibration_indices
            )
        )

        if overlap:

            raise ValueError(
                "Calibration/sample overlap detected!\n"
                f"Overlapping indices: {sorted(overlap)}"
            )

        print(
            "\nCalibration overlap check: PASS"
        )


# ============================================================
# 4. LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT_PATH):

    results_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    if "index" not in results_df.columns:

        raise ValueError(
            "Checkpoint exists but has no 'index' column."
        )

    results_df["index"] = (
        results_df["index"]
        .astype(str)
    )

    print()
    print("=" * 80)
    print("EXISTING CHECKPOINT FOUND")
    print("=" * 80)

    print(
        "Checkpoint rows:",
        len(results_df)
    )

else:

    results_df = pd.DataFrame()

    print()
    print("=" * 80)
    print("NO CHECKPOINT FOUND")
    print("=" * 80)

    print(
        "Starting from zero successful evaluations."
    )


# ============================================================
# 5. CLEAN CHECKPOINT
#
# CRITICAL:
# ONLY parse_status == "OK" COUNTS AS COMPLETED
# ============================================================

if not results_df.empty:

    # Remove rows that are not part of the fixed sample
    invalid_checkpoint_indices = (
        set(results_df["index"])
        - sample_indices
    )

    if invalid_checkpoint_indices:

        print()
        print(
            "WARNING: Removing checkpoint rows "
            "not belonging to the fixed sample:"
        )

        print(
            sorted(
                invalid_checkpoint_indices
            )
        )

        results_df = results_df[
            results_df["index"].isin(
                sample_indices
            )
        ].copy()


    # --------------------------------------------------------
    # Remove duplicate indices
    # --------------------------------------------------------

    results_df = (
        results_df
        .drop_duplicates(
            subset=["index"],
            keep="last"
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # CRITICAL FILTER
    #
    # Failed/API_ERROR rows are NOT completed.
    # --------------------------------------------------------

    if "parse_status" in results_df.columns:

        failed_checkpoint_rows = (
            results_df[
                results_df["parse_status"]
                != "OK"
            ]
        )

        if len(failed_checkpoint_rows) > 0:

            print()
            print(
                "Removing previously failed rows "
                "from completion checkpoint:"
            )

            print(
                failed_checkpoint_rows[
                    "index"
                ].tolist()
            )

            results_df = results_df[
                results_df["parse_status"]
                == "OK"
            ].copy()


# ============================================================
# 6. DETERMINE SUCCESSFULLY COMPLETED INDICES
# ============================================================

if results_df.empty:

    completed_indices = set()

else:

    completed_indices = set(
        results_df["index"]
        .astype(str)
    )


# Safety check
completed_indices = (
    completed_indices
    .intersection(sample_indices)
)


# ============================================================
# 7. DETERMINE REMAINING ROWS
# ============================================================

remaining_df = sample_df[
    ~sample_df["index"].isin(
        completed_indices
    )
].copy()


print()
print("=" * 80)
print("RESUME STATUS")
print("=" * 80)

print(
    "Total sample:",
    EVAL_SIZE
)

print(
    "Successfully completed:",
    len(completed_indices)
)

print(
    "Remaining:",
    len(remaining_df)
)


# ============================================================
# 8. EVALUATION LOOP
# ============================================================

rate_limit_hit = False
initial_completed_count = len(completed_indices)

for position, (_, row) in enumerate(
    remaining_df.iterrows(),
    start=1
):

    index = str(
        row["index"]
    )

    overall_number = (
        initial_completed_count
        + position
    )


    print()
    print("=" * 80)

    print(
        f"EVALUATING "
        f"{overall_number}/{EVAL_SIZE}"
    )

    print(
        f"Index: {index}"
    )

    print("=" * 80)


    result = None
    successful = False


    # ========================================================
    # RETRY LOOP
    # ========================================================

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            print(
                f"Attempt {attempt}/{MAX_RETRIES}"
            )


            # ------------------------------------------------
            # ONE JUDGE CALL
            # ------------------------------------------------

            result = judge_one_row(
                row,
                reasoning_effort=REASONING_EFFORT,
                max_retries=1
            )


            parse_status = (
                result.get(
                    "parse_status"
                )
            )


            # ------------------------------------------------
            # SUCCESS
            # ------------------------------------------------

            if parse_status == "OK":

                successful = True

                print(
                    "Parse: OK"
                )

                break


            # ------------------------------------------------
            # ERROR
            # ------------------------------------------------

            error_text = str(
                result.get(
                    "error",
                    ""
                )
            )

            raw_output = str(
                result.get(
                    "raw_output",
                    ""
                )
            )

            combined_error = (
                error_text
                + " "
                + raw_output
            )


            # ------------------------------------------------
            # RATE LIMIT
            # ------------------------------------------------

            if is_rate_limit_error(
                combined_error
            ):

                print()
                print(
                    "🚨 GROQ RATE/QUOTA LIMIT DETECTED"
                )

                print(
                    combined_error
                )

                rate_limit_hit = True

                break


            # ------------------------------------------------
            # NORMAL FAILURE
            # ------------------------------------------------

            print(
                f"Evaluation failed "
                f"(attempt {attempt})."
            )

            if attempt < MAX_RETRIES:

                print(
                    f"Retrying in "
                    f"{RETRY_WAIT} seconds..."
                )

                time.sleep(
                    RETRY_WAIT
                )


        except Exception as e:

            error_text = str(e)

            print()
            print(
                "Exception:"
            )

            print(
                error_text
            )


            # ------------------------------------------------
            # RATE LIMIT EXCEPTION
            # ------------------------------------------------

            if is_rate_limit_error(
                error_text
            ):

                print()
                print(
                    "🚨 GROQ RATE/QUOTA LIMIT DETECTED"
                )

                rate_limit_hit = True

                break


            if attempt < MAX_RETRIES:

                print(
                    f"Retrying in "
                    f"{RETRY_WAIT} seconds..."
                )

                time.sleep(
                    RETRY_WAIT
                )


    # ========================================================
    # STOP ON RATE LIMIT
    # ========================================================

    if rate_limit_hit:

        print()
        print("=" * 80)
        print("STOPPING EVALUATION")
        print("=" * 80)

        print(
            "Reason: Groq rate/token quota reached."
        )

        print(
            "All previously successful rows "
            "have already been checkpointed."
        )

        break


    # ========================================================
    # IF ROW DID NOT SUCCEED
    #
    # DO NOT ADD IT TO CHECKPOINT
    # ========================================================

    if not successful:

        print()
        print(
            "⚠️ ROW NOT SAVED AS COMPLETED"
        )

        print(
            "Index:",
            index
        )

        print(
            "Reason:",
            result.get(
                "parse_status"
            )
            if result
            else "No result"
        )


        # ----------------------------------------------------
        # Save failure separately
        # ----------------------------------------------------

        failure_row = {

            "index": index,

            "parse_status": (
                result.get(
                    "parse_status"
                )
                if result
                else None
            ),

            "error": (
                result.get(
                    "error"
                )
                if result
                else None
            ),

            "raw_output": (
                result.get(
                    "raw_output"
                )
                if result
                else None
            ),

            "finish_reason": (
                result.get(
                    "finish_reason"
                )
                if result
                else None
            ),

            "time_seconds": (
                result.get(
                    "time_seconds"
                )
                if result
                else None
            )
        }


        failure_df = pd.DataFrame(
            [failure_row]
        )


        if os.path.exists(
            FAILURE_LOG_PATH
        ):

            existing_failures = pd.read_csv(
                FAILURE_LOG_PATH
            )

            failure_df = pd.concat(
                [
                    existing_failures,
                    failure_df
                ],
                ignore_index=True
            )


        failure_df["index"] = (
            failure_df["index"]
            .astype(str)
        )


        failure_df = (
            failure_df
            .drop_duplicates(
                subset=["index"],
                keep="last"
            )
        )


        atomic_save_csv(
            failure_df,
            FAILURE_LOG_PATH
        )


        print(
            "Failure logged:",
            FAILURE_LOG_PATH
        )


        # Move to next row
        continue


    # ========================================================
    # BUILD SUCCESSFUL RESULT ROW
    # ========================================================

    result_row = {

        "index": index,

        "input": row["input"],

        "expected_answer":
            row["expected_answer"],

        "model_response":
            row["model_response"],

        "dataset":
            (
                row["dataset"]
                if "dataset"
                in row.index
                else None
            ),

        "task_type":
            (
                row["task_type"]
                if "task_type"
                in row.index
                else None
            ),

        "question_valid":
            result.get(
                "question_valid"
            ),

        "reference_answer_correct":
            result.get(
                "reference_answer_correct"
            ),

        "answer_correct":
            result.get(
                "answer_correct"
            ),

        "reasoning_correct":
            result.get(
                "reasoning_correct"
            ),

        "judge_evidence":
            result.get(
                "judge_evidence"
            ),

        "parse_status":
            result.get(
                "parse_status"
            ),

        "raw_output":
            result.get(
                "raw_output"
            ),

        "finish_reason":
            result.get(
                "finish_reason"
            ),

        "error":
            result.get(
                "error"
            ),

        "time_seconds":
            result.get(
                "time_seconds"
            )
    }


    # ========================================================
    # ADD SUCCESSFUL RESULT
    # ========================================================

    new_row_df = pd.DataFrame(
        [result_row]
    )


    if results_df.empty:

        results_df = (
            new_row_df.copy()
        )

    else:

        results_df = pd.concat(
            [
                results_df,
                new_row_df
            ],
            ignore_index=True
        )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    results_df["index"] = (
        results_df["index"]
        .astype(str)
    )


    # --------------------------------------------------------
    # Keep ONLY successful rows
    # --------------------------------------------------------

    results_df = results_df[
        results_df["parse_status"]
        == "OK"
    ].copy()


    # --------------------------------------------------------
    # Remove duplicates
    # --------------------------------------------------------

    results_df = (
        results_df
        .drop_duplicates(
            subset=["index"],
            keep="last"
        )
        .reset_index(drop=True)
    )


    # ========================================================
    # IMMEDIATE CHECKPOINT
    # ========================================================

    atomic_save_csv(
        results_df,
        CHECKPOINT_PATH
    )


    # Update completed set immediately
    completed_indices = set(
        results_df["index"]
        .astype(str)
    )


    print()
    print(
        "✅ SUCCESSFULLY SAVED"
    )

    print(
        "Index:",
        index
    )

    print(
        "Answer:",
        result.get(
            "answer_correct"
        )
    )

    print(
        "Reasoning:",
        result.get(
            "reasoning_correct"
        )
    )

    print(
        "Parse:",
        result.get(
            "parse_status"
        )
    )

    print(
        "Successful checkpoint rows:",
        len(results_df)
    )


    # ========================================================
    # SMALL PAUSE
    # ========================================================

    time.sleep(1)


# ============================================================
# 9. FINAL CHECKPOINT SAVE
# ============================================================

if not results_df.empty:

    atomic_save_csv(
        results_df,
        CHECKPOINT_PATH
    )


# ============================================================
# 10. FINAL STATUS
# ============================================================

print()
print("=" * 80)
print("CHECKPOINT STATUS")
print("=" * 80)

print(
    "Successfully completed:",
    len(results_df),
    "/",
    EVAL_SIZE
)

print(
    "Remaining:",
    EVAL_SIZE - len(results_df)
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH
)


# ============================================================
# 11. FINAL RESULTS ONLY AFTER ALL 300 SUCCESSFULLY PARSED
# ============================================================

if len(results_df) == EVAL_SIZE:

    atomic_save_csv(
        results_df,
        FINAL_RESULTS_PATH
    )

    print()
    print("=" * 80)
    print("🎯 ALL 300 ROWS SUCCESSFULLY EVALUATED")
    print("=" * 80)

    print(
        "Final results:",
        FINAL_RESULTS_PATH
    )

else:

    print()
    print("=" * 80)
    print("EVALUATION NOT COMPLETE")
    print("=" * 80)

    print(
        "The same cell can be run again "
        "after the quota resets."
    )


# ============================================================
# 12. CURRENT SUMMARY
# ============================================================

if not results_df.empty:

    print()
    print("=" * 80)
    print("CURRENT SUCCESSFUL RESULTS")
    print("=" * 80)


    print("\nParse status:")

    print(
        results_df[
            "parse_status"
        ].value_counts(
            dropna=False
        )
    )


    successful = results_df[
        results_df["parse_status"]
        == "OK"
    ].copy()


    valid_cases = successful[
        (
            successful[
                "question_valid"
            ]
            == True
        )
        &
        (
            successful[
                "reference_answer_correct"
            ]
            == True
        )
    ].copy()


    print(
        "\nSuccessfully judged:",
        len(successful)
    )

    print(
        "Valid evaluation cases:",
        len(valid_cases)
    )


    if len(valid_cases) > 0:

        answer_accuracy = (
            valid_cases[
                "answer_correct"
            ].mean()
            * 100
        )

        reasoning_accuracy = (
            valid_cases[
                "reasoning_correct"
            ].mean()
            * 100
        )


        print(
            f"\nCurrent answer accuracy: "
            f"{answer_accuracy:.2f}%"
        )

        print(
            f"Current reasoning accuracy: "
            f"{reasoning_accuracy:.2f}%"
        )

In [1]:
import pandas as pd

checkpoint_path = "/kaggle/working/llama_groq_final_300_checkpoint.csv"

df = pd.read_csv(checkpoint_path)

print("Rows:", len(df))
print("Unique indices:", df["index"].nunique())
print("\nParse status:")
print(df["parse_status"].value_counts())

Rows: 31
Unique indices: 31

Parse status:
parse_status
OK    31
Name: count, dtype: int64


In [5]:
import os
import shutil
from IPython.display import FileLink, display

checkpoint_path = "/kaggle/working/llama_groq_final_300_checkpoint.csv"

zip_base = "/kaggle/working/llama_groq_final_300_checkpoint"

# Create ZIP
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir="/kaggle/working",
    base_dir="llama_groq_final_300_checkpoint.csv"
)

print("ZIP created:")
print(zip_path)

print("\nExists:", os.path.exists(zip_path))
print(
    "Size:",
    round(os.path.getsize(zip_path) / 1024, 2),
    "KB"
)

display(
    FileLink(
        "llama_groq_final_300_checkpoint.zip"
    )
)

ZIP created:
/kaggle/working/llama_groq_final_300_checkpoint.zip

Exists: True
Size: 6.66 KB


/kaggle/working/llama_groq_final_300_checkpoint.zip

In [4]:
import os

print(
    os.path.exists(
        "/kaggle/working/"
        "llama_groq_random_300_checkpoint.csv"
    )
)

False
